# Benefits Market Intelligence Exploratory Data Analysis: Form 5500

## Libraries

In [ ]:
# Libraries
import pandas as pd
import numpy as np
import duckdb
import plotly.express as px
import re
from benefits_market_intelligence.config.paths import DB_PATH, FIGURES_PATH

In [ ]:
# Displaying all
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

## Load Form 5500 data

In [ ]:
# Loading data
with duckdb.connect(DB_PATH, read_only=True) as con:
    F_5500 = con.sql("SELECT * FROM silver.F_5500").df()

In [ ]:
# Convert pandas-made object columns to string
obj_cols = F_5500.dtypes[F_5500.dtypes == "object"].index
F_5500[obj_cols] = F_5500[obj_cols].astype("str")

## Viewing data

In [ ]:
# Head of F5500
F_5500.head()

## Shape of data

In [ ]:
# Shape
print(f"Rows: {F_5500.shape[0]}\nColumns: {F_5500.shape[1]}")

## Data types

In [ ]:
# Data type counts
F_5500.dtypes.value_counts()

## Any missing data?

In [ ]:
# Columns with no missing data
F_5500.isna().sum()[F_5500.isna().sum() == 0]

In [ ]:
# Missing data
pd.DataFrame(
    {
        "missing_count": F_5500.isna().sum(),
        "missing_percent": F_5500.isna().mean().mul(100),
    }
).query("missing_count > 0").sort_values(
    "missing_percent", ascending=False
).rename_axis("column_name").reset_index()

## Plotly setup

In [ ]:
# Plotly setup and figure-saving helper
FIGURES_PATH.mkdir(parents=True, exist_ok=True)


def _find_col(candidates=(), include_terms=(), exclude_terms=()):
    """Find the first useful column from exact candidates, then keyword matches."""
    cols = list(F_5500.columns)
    normalized = {
        c: re.sub(r"[^a-z0-9]+", "_", str(c).lower()).strip("_") for c in cols
    }

    for candidate in candidates:
        cand = re.sub(r"[^a-z0-9]+", "_", candidate.lower()).strip("_")
        for c, n in normalized.items():
            if n == cand:
                return c

    for c, n in normalized.items():
        if all(term.lower() in n for term in include_terms) and not any(
            term.lower() in n for term in exclude_terms
        ):
            return c
    return None


def save_figure(fig, filename):
    path = FIGURES_PATH / f"{filename}.png"
    fig.show()
    fig.write_image(path, scale=2)


# Common Form 5500 fields. The fallback matching makes the notebook resilient
# to small naming differences in the underlying silver table.
YEAR_COL = _find_col(
    ["FORM_PLAN_YEAR_BEGIN_DATE", "PLAN_YEAR_BEGIN_DATE", "FORM_YEAR"],
    include_terms=("plan", "year"),
)
STATE_COL = _find_col(
    ["SPONS_DFE_MAIL_US_STATE", "SPONSOR_STATE", "STATE"],
    include_terms=("state",),
    exclude_terms=("participant",),
)
SPONSOR_COL = _find_col(
    ["SPONS_DFE_NAME", "SPONSOR_NAME", "SPONSOR_DFE_NAME"],
    include_terms=("spons", "name"),
)
PLAN_TYPE_COL = _find_col(
    ["TYPE_PLAN_ENTITY_CD", "PLAN_ENTITY_TYPE", "TYPE_PLAN_FILING"],
    include_terms=("type", "plan"),
)
PARTICIPANTS_COL = _find_col(
    ["TOT_ACTIVE_PARTCP_CNT", "TOT_PARTCP_BNFT_CNT", "TOTAL_PARTICIPANTS"],
    include_terms=("partcp",),
    exclude_terms=("employer",),
)
ASSETS_EOY_COL = _find_col(
    ["TOT_ASSETS_EOY_AMT", "TOTAL_ASSETS_EOY", "ASSETS_EOY"],
    include_terms=("assets", "eoy"),
)
ASSETS_BOY_COL = _find_col(
    ["TOT_ASSETS_BOY_AMT", "TOTAL_ASSETS_BOY", "ASSETS_BOY"],
    include_terms=("assets", "boy"),
)
EMPLOYER_CONTRIB_COL = _find_col(
    ["TOT_CONTRIB_EMPLR_AMT", "TOTAL_EMPLOYER_CONTRIBUTIONS"],
    include_terms=("contrib", "emplr"),
)
PARTICIPANT_CONTRIB_COL = _find_col(
    ["TOT_CONTRIB_PRTCP_AMT", "TOTAL_PARTICIPANT_CONTRIBUTIONS"],
    include_terms=("contrib", "prtcp"),
)

print("Fields selected:")
for label, col in {
    "Year": YEAR_COL,
    "State": STATE_COL,
    "Sponsor": SPONSOR_COL,
    "Plan type": PLAN_TYPE_COL,
    "Participants": PARTICIPANTS_COL,
    "EOY assets": ASSETS_EOY_COL,
    "BOY assets": ASSETS_BOY_COL,
    "Employer contributions": EMPLOYER_CONTRIB_COL,
    "Participant contributions": PARTICIPANT_CONTRIB_COL,
}.items():
    print(f"  {label}: {col}")

## Filing volume over time

In [ ]:
# Annual filing volume — useful for sizing the addressable market and monitoring growth.
if YEAR_COL is not None:
    year = pd.to_datetime(F_5500[YEAR_COL], errors="coerce").dt.year
    annual_filings = (
        year.dropna()
        .astype(int)
        .value_counts()
        .sort_index()
        .rename_axis("year")
        .reset_index(name="filings")
    )

    fig = px.line(
        annual_filings,
        x="year",
        y="filings",
        markers=True,
        title="Form 5500 Filing Volume Over Time",
        labels={"year": "Plan year", "filings": "Number of filings"},
    )
    fig.update_traces(hovertemplate="Plan year: %{x}<br>Filings: %{y:,}<extra></extra>")
    fig.update_layout(hovermode="x unified")
    save_figure(fig, "filing_volume_over_time")
else:
    print("Skipped: no plan-year field was found.")

## Geographic concentration of sponsors

In [ ]:
# Sponsor geography — highlights states with the largest pools of potential prospects.
if STATE_COL is not None:
    state_counts = (
        F_5500[STATE_COL]
        .replace({"nan": np.nan, "": np.nan})
        .dropna()
        .value_counts()
        .head(20)
        .sort_values()
        .rename_axis("state")
        .reset_index(name="filings")
    )

    fig = px.bar(
        state_counts,
        x="filings",
        y="state",
        orientation="h",
        title="Top States by Form 5500 Filing Volume",
        labels={"state": "Sponsor state", "filings": "Number of filings"},
        text="filings",
    )
    fig.update_traces(texttemplate="%{text:,}", textposition="outside")
    fig.update_layout(yaxis=dict(categoryorder="total ascending"))
    save_figure(fig, "top_states_by_filing_volume")
else:
    print("Skipped: no sponsor-state field was found.")

## Plan/entity type mix

In [ ]:
# Filing mix by plan/entity type — useful for understanding the composition of the market.
if PLAN_TYPE_COL is not None:
    type_counts = (
        F_5500[PLAN_TYPE_COL]
        .replace({"nan": np.nan, "": np.nan})
        .dropna()
        .value_counts()
        .head(15)
        .sort_values(ascending=True)
        .rename_axis("plan_type")
        .reset_index(name="filings")
    )

    fig = px.bar(
        type_counts,
        x="filings",
        y="plan_type",
        orientation="h",
        title="Form 5500 Filings by Plan/Entity Type",
        labels={"plan_type": "Plan/entity type", "filings": "Number of filings"},
        text="filings",
    )
    fig.update_traces(texttemplate="%{text:,}", textposition="outside")
    fig.update_layout(yaxis=dict(categoryorder="total ascending"))
    save_figure(fig, "filing_mix_by_plan_type")
else:
    print("Skipped: no plan/entity-type field was found.")

## Sponsor concentration

In [ ]:
# Sponsor concentration — identifies organizations appearing across multiple filings.
if SPONSOR_COL is not None:
    sponsor_counts = (
        F_5500[SPONSOR_COL]
        .replace({"nan": np.nan, "": np.nan})
        .dropna()
        .value_counts()
        .head(20)
        .sort_values()
        .rename_axis("sponsor")
        .reset_index(name="filings")
    )

    fig = px.bar(
        sponsor_counts,
        x="filings",
        y="sponsor",
        orientation="h",
        title="Organizations with the Most Form 5500 Filings",
        labels={"sponsor": "Sponsor / organization", "filings": "Number of filings"},
        text="filings",
    )
    fig.update_traces(texttemplate="%{text:,}", textposition="outside")
    fig.update_layout(yaxis=dict(categoryorder="total ascending"))
    save_figure(fig, "sponsor_concentration")
else:
    print("Skipped: no sponsor-name field was found.")